# 02 · Run artifacts — what ran, and what it produced

A run that leaves nothing behind cannot be inspected, compared, or argued with.
This notebook builds the artifact store the rest of the repo reads from: three
JSON files per run — `manifest.json`, `papers.json`, `answer.json` — written
under `runs/<run_id>/`, then read back with `nbio.load_run()` and
`nbio.list_run_ids()`.

Ported from the private product's
`clinical_search/services/run_store.py`, with the S3 mirror and the local-cleanup
path left out (see *What did not come across*).

This is the **artifacts** level of observability. Counting
(`01-cost-and-budget.ipynb`) tells you what a run spent; artifacts tell you what
it produced. Neither tells you why it chose what it chose — that is
`03-tracing.ipynb`.

**No API key is needed.** Nothing here calls a model; it writes and reads JSON.
Every run id created below starts with `demo-observe-`, and `runs/` is
gitignored, so nothing written here is committed.

## What this notebook demonstrates

| Name | What it does | One example |
|---|---|---|
| `new_run_id` | A collision-free id for one run | `new_run_id()` -> a uuid4 string |
| `save_manifest` / `save_papers` / `save_answer` | Write the three artifacts of one run, atomically per file | `save_manifest(rid, {"query": ..., "model_id": ...})` |
| `nbio.runs_dir` | The one directory every artifact is written under | `nbio.runs_dir() / run_id / "manifest.json"` |
| `nbio.list_run_ids` | Every run id on disk, oldest first, by mtime | `nbio.list_run_ids()[-1]` is the newest |
| `nbio.load_run` | Reads a run's three artifacts back as one dict | `nbio.load_run("demo-observe-artifacts")["manifest"]` |
| `nbio.pages` | Per-item records for stages that write one file per document | `nbio.pages(rid, "extract")` |
| `manifest_summary` | One row per run, for a list view | `manifest_summary(rid, manifest)["paper_count"]` |
| The read-don't-restate rule | A displayed number is read from the artifact, never typed inline | Step 8 shows a restated number going stale |

## Step 1 — locate the repo root and import `nbio`

Jupyter starts a kernel with its working directory set to the notebook's own
folder, two levels below the repo root, so a bare `import nbio` fails. Walk up
until `nbio.py` is found, then import it.

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()

## Step 2 — where artifacts go, and what one run looks like on disk

`nbio.runs_dir()` resolves to `runs/` at the repo root — one directory per run,
named by run id, holding up to three files. Every path below is printed relative
to the repo root: an absolute path on someone else's machine is noise, and
absolute paths have leaked into this repo through committed notebook outputs
before.

In [ ]:
repo_root = nbio.bootstrap()
RUNS = nbio.runs_dir()

DEMO_RUN = "demo-observe-artifacts"     # clearly demo-named: this is not a real run
ARTIFACT_FILES = ("manifest.json", "papers.json", "answer.json")

# Re-running this notebook should start from a clean demo state, so anything a
# previous run left behind under these ids goes first. Nothing outside
# runs/demo-observe-artifacts* is touched.
import shutil
for stale in sorted(RUNS.glob("demo-observe-artifacts*")):
    if stale.is_dir():
        shutil.rmtree(stale)

print("runs directory :", RUNS.relative_to(repo_root))
print("this run       :", (RUNS / DEMO_RUN).relative_to(repo_root))
for name in ARTIFACT_FILES:
    print("   ", (RUNS / DEMO_RUN / name).relative_to(repo_root).name)

assert RUNS.parent == repo_root
assert RUNS.name == "runs"

## Step 3 — the writer, ported

Four functions and a helper. Each `save_*` creates the run directory if it does
not exist and writes one file, so a run that dies halfway still leaves the
artifacts it got to — a partial run you can read beats a complete run you can't.

The product version calls an S3 mirror after each write and can delete the local
copy once the upload is confirmed. Both are dropped here: this repo's readers all
read local files, and a cookbook that silently deletes what it just wrote would
teach the wrong lesson.

In [ ]:
import json as jsonlib
import uuid
from datetime import datetime, timezone


def new_run_id() -> str:
    return str(uuid.uuid4())


def _write_json(path: Path, data) -> None:
    path.write_text(jsonlib.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def _run_dir(run_id: str) -> Path:
    # Everything this notebook writes stays under runs/ and nowhere else.
    d = nbio.runs_dir() / run_id
    d.mkdir(parents=True, exist_ok=True)
    return d


def save_manifest(run_id: str, data: dict) -> None:
    _write_json(_run_dir(run_id) / "manifest.json", data)


def save_papers(run_id: str, papers: list) -> None:
    _write_json(_run_dir(run_id) / "papers.json", {"run_id": run_id, "papers": papers})


def save_answer(run_id: str, data: dict) -> None:
    _write_json(_run_dir(run_id) / "answer.json", data)


print("writer ready:", [f.__name__ for f in (new_run_id, save_manifest, save_papers, save_answer)])
print("sample run id:", new_run_id())

assert len(new_run_id()) == 36 and new_run_id() != new_run_id()

## Step 4 — the manifest: what ran

The manifest is the run's identity card — the query, the model, the timing, and
the counts. It is what a list view reads, and it is the only artifact every
downstream reader can assume exists. Note `duration_ms` and `cost_usd` sitting
in here: the counting level from notebook 01 lands in the artifact, which is how
a cost stops being a number that scrolled past in a cell.

In [ ]:
started = datetime(2026, 9, 17, 14, 30, 0, tzinfo=timezone.utc)
completed = datetime(2026, 9, 17, 14, 30, 8, tzinfo=timezone.utc)

PAPERS = [
    {"id": "P1", "title": "Enzymatic debridement in partial-thickness burns", "year": 2021, "score": 0.82},
    {"id": "P2", "title": "Early excision timing and graft take", "year": 2019, "score": 0.77},
    {"id": "P3", "title": "Dressing choice in paediatric scalds", "year": 2023, "score": 0.61},
]

manifest = {
    "run_id": DEMO_RUN,
    "query": "enzymatic debridement in partial-thickness burns",
    "model_id": "openai/gpt-oss-120b",
    "model_provider": "groq",
    "framework": "cookbook-demo",
    "started_at": started.isoformat(),
    "completed_at": completed.isoformat(),
    "duration_ms": int((completed - started).total_seconds() * 1000),
    "paper_count": len(PAPERS),
    "cost_usd": 0.000612,
}

save_manifest(DEMO_RUN, manifest)
nbio.show_json(manifest)

assert (nbio.runs_dir() / DEMO_RUN / "manifest.json").is_file()

## Step 5 — the other two artifacts: what it retrieved, what it said

`papers.json` is the evidence the run had in hand; `answer.json` is what it did
with it. Keeping them apart is what makes a retrieval failure distinguishable
from a generation failure after the fact — the same split `01-tools/06-bench`'s
gold-context arm relies on.

In [ ]:
save_papers(DEMO_RUN, PAPERS)
save_answer(DEMO_RUN, {
    "run_id": DEMO_RUN,
    "answer": "Enzymatic debridement reduced time to wound closure in the retrieved cohort studies.",
    "citations": ["P1", "P2"],
    "grounded": True,
})

written = sorted(p.name for p in (nbio.runs_dir() / DEMO_RUN).glob("*.json"))
print("files on disk for this run:", written)

assert written == ["answer.json", "manifest.json", "papers.json"]

## Step 6 — reading it back with `nbio.load_run()` and `nbio.list_run_ids()`

`list_run_ids()` sorts by mtime, **oldest first** — run ids are uuid4 or
ad-hoc timestamps, so lexicographic order tells you nothing about recency, and
`[-1]` without that sort would hand you an arbitrary run. `load_run(run_id)`
returns `{"run_id", "manifest", "papers", "answer"}`, leaving any missing
artifact as `None` rather than raising.

In [ ]:
ids = nbio.list_run_ids()
print(f"{len(ids)} run(s) on disk; newest last:")
for rid in ids[-5:]:
    print("  ", rid)

loaded = nbio.load_run(DEMO_RUN)
print()
print("keys returned by load_run:", sorted(loaded))
print("query     :", loaded["manifest"]["query"])
print("papers    :", len(loaded["papers"]["papers"]))
print("answer    :", loaded["answer"]["answer"][:60] + "...")

assert DEMO_RUN in ids
assert loaded["manifest"] == manifest             # byte-for-byte what was written
assert loaded["papers"]["papers"] == PAPERS
assert nbio.load_run("no-such-run-id")["manifest"] is None   # missing, not an exception

## Step 7 — the rule: read the number, never restate it

A number displayed in a notebook should be **read from the run artifact**, never
typed inline. This is the whole discipline of the artifacts level, and it is the
reason `nbio.load_run` exists at all.

The inline version is not wrong when you write it. It is wrong three weeks later,
after the pipeline changed and nobody re-ran the prose. The artifact version
cannot go stale, because it is not a claim about the run — it *is* the run.

In [ ]:
run = nbio.load_run(DEMO_RUN)

# Read from the artifact. There is no second copy of these numbers anywhere.
print(f"query       : {run['manifest']['query']}")
print(f"papers      : {run['manifest']['paper_count']}")
print(f"duration    : {run['manifest']['duration_ms']} ms")
print(f"cost        : ${run['manifest']['cost_usd']:.6f}")
print(f"citations   : {len(run['answer']['citations'])} of {len(run['papers']['papers'])} papers cited")

# The count in the manifest and the actual list length are checked against each
# other -- an artifact can disagree with itself too.
assert run["manifest"]["paper_count"] == len(run["papers"]["papers"])
assert all(c in {p["id"] for p in run["papers"]["papers"]} for c in run["answer"]["citations"])

## Step 8 — drift, demonstrated

Below, a notebook author writes "the pipeline retrieved 3 papers" inline, and
alongside it reads the same figure from the artifact. Then the pipeline is
re-run with a widened search and writes 5 papers.

The restated number does not change. It is now simply false, and nothing in the
notebook will ever tell you so. The read number follows the run without anyone
touching the cell.

In [ ]:
RESTATED_PAPER_COUNT = 3          # typed by hand while looking at the first run
read_before = nbio.load_run(DEMO_RUN)["manifest"]["paper_count"]
print(f"before re-run  -- restated: {RESTATED_PAPER_COUNT}   read: {read_before}   agree: "
      f"{RESTATED_PAPER_COUNT == read_before}")

# The pipeline is re-run with a wider search and overwrites the same run id.
WIDER_PAPERS = PAPERS + [
    {"id": "P4", "title": "Hydrosurgical debridement versus enzymatic", "year": 2022, "score": 0.58},
    {"id": "P5", "title": "Cost analysis of burn debridement pathways", "year": 2020, "score": 0.44},
]
save_papers(DEMO_RUN, WIDER_PAPERS)
save_manifest(DEMO_RUN, {**manifest, "paper_count": len(WIDER_PAPERS), "duration_ms": 11_400})

read_after = nbio.load_run(DEMO_RUN)["manifest"]["paper_count"]
print(f"after re-run   -- restated: {RESTATED_PAPER_COUNT}   read: {read_after}   agree: "
      f"{RESTATED_PAPER_COUNT == read_after}")
print()
print(f"the restated number is now wrong by {read_after - RESTATED_PAPER_COUNT}, silently")
print(f"the read number is still correct, and nobody edited the cell that prints it")

assert read_before == 3
assert read_after == 5
assert RESTATED_PAPER_COUNT != read_after                              # the restated one drifted
assert read_after == len(nbio.load_run(DEMO_RUN)["papers"]["papers"])   # the read one did not

## Step 9 — many runs: the list view

The product's `list_runs()` reads every manifest on disk and flattens each to one
summary row. That projection is the reason a manifest keeps `query`, `model_id`
and `duration_ms` at the top level instead of nesting them: a list view should
never have to open `papers.json` to render a row.

A second demo run is written here so the list has something to compare.

In [ ]:
SECOND_RUN = "demo-observe-artifacts-b"
save_manifest(SECOND_RUN, {
    **manifest,
    "run_id": SECOND_RUN,
    "query": "early excision timing in paediatric scalds",
    "model_id": "gpt-4o-mini",
    "model_provider": "openai",
    "started_at": datetime(2026, 9, 17, 15, 2, 0, tzinfo=timezone.utc).isoformat(),
    "duration_ms": 6_200,
    "paper_count": 2,
    "cost_usd": 0.001004,
})
save_papers(SECOND_RUN, PAPERS[:2])


def manifest_summary(run_id: str, manifest: dict) -> dict:
    """One run flattened to a row -- the projection a list view renders."""
    return {
        "run_id": run_id,
        "query": manifest.get("query", ""),
        "model_id": manifest.get("model_id", ""),
        "started_at": manifest.get("started_at", ""),
        "duration_ms": manifest.get("duration_ms"),
        "paper_count": manifest.get("paper_count", 0),
        "cost_usd": manifest.get("cost_usd", 0.0),
    }


def list_runs(prefix: str = "demo-observe") -> list:
    """Summaries for runs on disk, newest first, read from their manifests."""
    out = []
    for rid in reversed(nbio.list_run_ids()):
        if not rid.startswith(prefix):
            continue
        run = nbio.load_run(rid)
        if run and run["manifest"]:
            out.append(manifest_summary(rid, run["manifest"]))
    return out


summaries = list_runs()
nbio.table(
    [(s["run_id"], s["model_id"], s["paper_count"], s["duration_ms"], f"${s['cost_usd']:.6f}")
     for s in summaries],
    ("run id", "model", "papers", "ms", "cost"),
)

assert len(summaries) >= 2
assert {s["run_id"] for s in summaries} >= {DEMO_RUN, SECOND_RUN}
assert all(s["paper_count"] == len(nbio.load_run(s["run_id"])["papers"]["papers"]) for s in summaries)

## Step 10 — one file per item, when a bundle is the wrong shape

Extraction and chunking produce one record per page or per document, not one
bundle per run. For those, artifacts go under `runs/<run_id>/<stage>/<key>.json`
and are read with `nbio.page(run_id, stage, key)` for one, or `nbio.pages(run_id,
stage)` for all of them. Same directory, same discipline, different granularity —
a 400-page extraction in a single JSON file is a file nobody can diff.

In [ ]:
STAGE = "extract"
stage_dir = nbio.runs_dir() / DEMO_RUN / STAGE
stage_dir.mkdir(parents=True, exist_ok=True)

for i, paper in enumerate(WIDER_PAPERS[:3], start=1):
    _write_json(stage_dir / f"page-{i:03d}.json", {
        "page_key": f"page-{i:03d}",
        "source_id": paper["id"],
        "char_count": 1200 + i * 130,
    })

every = nbio.pages(DEMO_RUN, STAGE)
one = nbio.page(DEMO_RUN, STAGE, "page-002")
print(f"{len(every)} page record(s) written; page-002 reads back as:")
nbio.show_json(one)

assert len(every) == 3
assert one["source_id"] == WIDER_PAPERS[1]["id"]
assert [p["page_key"] for p in every] == ["page-001", "page-002", "page-003"]

## Step 11 — what the demo left on disk

`runs/` is gitignored, so none of this is committed, but it is still sitting in
your working copy and it is still the newest thing there — which matters,
because `nbio.load_run()` with no argument returns the most recent run. Delete
the demo runs when you are done if you want another notebook's bare
`load_run()` to find its own run again.

In [ ]:
demo_ids = [r for r in nbio.list_run_ids() if r.startswith("demo-observe")]
print("demo runs written by this notebook:")
for rid in demo_ids:
    files = sorted(p.name for p in (nbio.runs_dir() / rid).rglob("*.json"))
    print(f"  {rid}: {len(files)} file(s)")

print()
print("bare nbio.load_run() currently returns:", nbio.load_run()["run_id"])
print()
print("to clean up:")
print("  import shutil")
print("  for rid in [r for r in nbio.list_run_ids() if r.startswith('demo-observe')]:")
print("      shutil.rmtree(nbio.runs_dir() / rid)")

assert set(demo_ids) >= {DEMO_RUN, SECOND_RUN}
assert nbio.load_run()["run_id"].startswith("demo-observe")   # newest by mtime is ours

## Where this runs in the pipeline

At the end of every stage that produces something worth keeping.
`01-tools/03-embed` already writes an `index_manifest.json` per run this way, and
`06-bench` reads stored score components back out to re-rank with no API calls at
all — the cheapest experiment in the repo, and it is only possible because the
components were written down the first time.

## What did not come across

- **The S3 mirror.** `_mirror_to_s3()` uploads each artifact after it is written
  and `list_runs()` merges local and remote ids. Deployment machinery, and it
  needs credentials this repo will not have.
- **Local cleanup after upload.** The product deletes the local run directory
  once all three files are confirmed in S3. A cookbook that deleted what it just
  taught you to write would be actively unhelpful.
- **Atomic writes.** `_write_json` here is a plain `write_text`. A crash mid-write
  leaves a truncated file that `json.loads` rejects. Write to a temp file in the
  same directory and `os.replace` it if you are writing artifacts from something
  that can be killed.
- **Schema versioning.** Nothing in these files records which version of the
  pipeline produced them, so an old run and a new run are indistinguishable when
  the manifest shape changes. The product handles this with `.get(...)` defaults
  everywhere, which is tolerance, not versioning.
- **Retention.** Runs accumulate forever. There is no size cap, no expiry, and no
  index — `list_run_ids()` stats every directory under `runs/` on every call.